# 03. Data Cleaning & Preprocessing Notebook
**Owner:** Ravi (Data Cleaning & EDA Engineer)  
**Objective:** Interactive step-by-step cleaning of the Telco Customer Churn dataset, missing value imputation, duplicate checks, and binary target encoding.

In [6]:
import os
import pandas as pd
import numpy as np

RAW_PATH = os.path.join("..", "data", "raw", "WA_Fn-UseC_-Telco-Customer-Churn.csv")
ALT_RAW_PATH = os.path.join("..", "data", "raw", "telco_customer_churn.csv")
PROCESSED_PATH = os.path.join("..", "data", "processed", "customer_features.csv")

### 1. Load Raw Telecom Dataset
Loading raw customer records into a Pandas DataFrame and inspecting raw dataset shape.

In [7]:
raw_file = RAW_PATH if os.path.exists(RAW_PATH) else ALT_RAW_PATH
df = pd.read_csv(raw_file)
print(f"[INFO] Loaded raw dataset shape: {df.shape}")
df.head(3)

[INFO] Loaded raw dataset shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


### 2. Missing Value Imputation & Data Type Casting
Converting `TotalCharges` from text string to float numeric and imputing missing total charges for new customers (`tenure == 0`).

In [8]:
df = df.drop_duplicates(subset=["customerID"])

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"].astype(str).str.strip(), errors="coerce")

missing_count = df["TotalCharges"].isnull().sum()
if missing_count > 0:
    print(f"[INFO] Found {missing_count} missing values in TotalCharges. Imputing...")
    df["TotalCharges"] = df["TotalCharges"].fillna(df["MonthlyCharges"] * df["tenure"])

print("[SUCCESS] TotalCharges successfully converted to numeric with zero nulls!")

[INFO] Found 11 missing values in TotalCharges. Imputing...
[SUCCESS] TotalCharges successfully converted to numeric with zero nulls!


### 3. Binary Encoding for Target Variable (`Churn`)
Converting text target 'Yes' / 'No' into binary numbers `1` / `0` for Machine Learning algorithms.

In [4]:
df["ChurnBinary"] = df["Churn"].apply(lambda x: 1 if str(x).strip().lower() == "yes" else 0)
print("[SUCCESS] Binary target column 'ChurnBinary' created!")
df[["customerID", "MonthlyCharges", "TotalCharges", "tenure", "Churn", "ChurnBinary"]].head(5)

[SUCCESS] Binary target column 'ChurnBinary' created!


,customerID,MonthlyCharges,TotalCharges,tenure,Churn,ChurnBinary
0,7590-VHVEG,29.85,29.85,1,No,0
1,5575-GNVDE,56.95,1889.50,34,No,0
2,3668-QPYBK,53.85,108.15,2,Yes,1
3,7795-CFOCW,42.30,1840.75,45,No,0
4,9237-HQITU,70.70,151.65,2,Yes,1


### 4. Save Processed Dataset for Team
Exporting the final clean dataset to `data/processed/customer_features.csv`.

In [5]:
os.makedirs(os.path.dirname(PROCESSED_PATH), exist_ok=True)
df.to_csv(PROCESSED_PATH, index=False)
print(f"[SUCCESS] Processed dataset saved to: {PROCESSED_PATH}")

[SUCCESS] Processed dataset saved to: ..\data\processed\customer_features.csv
